# 低段位期・高段位期を分類するLightGBMの比較
M1からM4を順に実行します。C5の対局別カウンタ、第1段階の分割、第2段階の窓別集計を照合します。主分析は非重複10対局窓、感度分析は20対局窓です。

Aは既存20項目、Bは学習期間内で安定と判定した既存項目、CはA＋新3項目、DはB＋新3項目です。しょぼんｎの同じ将来区間でC−AとD−Bを比較します。多数派予測も併記します。

In [ ]:
# M1　Driveを接続し、固定コミットのコードと入力監査テストを準備
from google.colab import drive
from pathlib import Path
import sys, os, subprocess, tempfile, urllib.request, importlib.util
drive.mount('/content/drive')
PIN = '21b5b2c409daee1c916a5f2bfce34feff75d5973'
src = Path(tempfile.mkdtemp(prefix='mahjong_model_'))
files = ['mahjong_research/'+f for f in ('__init__.py','analysis.py','inputs.py','tenhou.py','legacy.py')]
files += ['smoothing/study.py','smoothing/presentation.py','smoothing/test_smoothing.py',
          'stability/stability.py','choices/choices.py']
files += ['model_comparison/'+f for f in ('windows.py','compare.py','outputs.py','test_comparison.py')]
for rel in files:
    target=src/rel
    target.parent.mkdir(exist_ok=True)
    url=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{PIN}/research_v2/{rel}'
    with urllib.request.urlopen(url,timeout=60) as r: target.write_bytes(r.read())
if importlib.util.find_spec('lightgbm') is None:
    subprocess.run([sys.executable,'-m','pip','install','lightgbm==4.6.0'],check=True)
paths=[str(src/p) for p in ('model_comparison','stability','smoothing','choices','.')]
env=dict(os.environ,PYTHONPATH=os.pathsep.join(paths),MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','unittest','-q',
                'test_comparison.ComparisonTests.test_saved_phase1_and_phase2_assembly',
                'test_comparison.ComparisonTests.test_selection_is_past_only',
                'test_comparison.ComparisonTests.test_single_class_and_precision'],cwd=src,env=env,check=True)

In [ ]:
# M2　入力を指定。第2段階の候補が複数あればQ5完了時の保存先を指定
ROOT = Path('/content/drive/MyDrive/卒論')
RUN = ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'
PHASE1 = ROOT/'research_v2_stability/20261105T092459Z_113b116b'
PHASE2 = None  # ROOT/'research_v2_choices/日時付きフォルダ名'
RESUME = None  # 中断再開時はM3に表示された出力先
ULTRA_CACHE = None  # 移動した場合のみウルトラ立直のC8完了キャッシュを指定
OUTPUT = ROOT/'research_v2_model_comparison'
if PHASE2 is None:
    choices=sorted(p.parent for p in (ROOT/'research_v2_choices').glob('*/window_values.csv')
                   if (p.parent/'new_game_features.csv').is_file())
    if len(choices)!=1:
        print(*choices,sep='\n')
        raise ValueError('Q5まで完了した実行フォルダをPHASE2に指定してください。')
    PHASE2=choices[0]
print('第1段階',PHASE1,'第2段階',PHASE2,'出力の親',OUTPUT,sep='\n')

In [ ]:
# M3　学習期間だけで特徴量を選び、4モデルの学習・内部評価・外部評価を実行
import json
args=[str(p) for p in (RUN,PHASE1,PHASE2,OUTPUT)]
args += [str(RESUME) if RESUME else None,str(ULTRA_CACHE) if ULTRA_CACHE else None]
code = """import json,sys
from pathlib import Path
from compare import run
out=run(*json.loads(sys.argv[1]))
Path(sys.argv[2]).write_text(str(out),encoding='utf8')
"""
target=src/'completed_path.txt'
subprocess.run([sys.executable,'-u','-c',code,json.dumps(args),str(target)],cwd=src,env=env,check=True)
RESULT=Path(target.read_text(encoding='utf8'))
print('保存先',RESULT)

In [ ]:
# M4　同一評価窓の差と各Foldの成立条件を確認
from IPython.display import display,Markdown,Image
import pandas as pd
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf8')))
display(pd.read_csv(RESULT/'fold_audit.csv'))
display(pd.read_csv(RESULT/'model_comparison.csv'))
display(Image(filename=str(RESULT/'performance_comparison.png')))

外部評価には事前指定したFold 3の固定モデルA・Bを使用します。C・Dに必要な新3項目は第2段階でしょぼんｎについてのみ抽出済みです。単一クラスの期間ではROC-AUC・Balanced Accuracyが欠測となります。

特徴量重要度はgain、および評価窓で特徴量1列を循環移動したときのAUC低下で確認します。入力列順と学習済みモデルは `models/`、予測確率は `predictions.csv` に保存します。中断後は同じ入力・ライブラリ版でM2の `RESUME` に実行フォルダを指定してください。